# Tema 04: Plotly

**Taller:** Análisis y Visualización Interactiva en Python
**Duración estimada de esta sesión:** 60 minutos
**Herramienta principal:** Plotly Express y Graph Objects
**Modalidad de práctica:** Google Colab

---

> 📌 **Nota para el profesor:** esta notebook está diseñada para proyectarse y ejecutarse en vivo. Las secciones marcadas como **Práctica guiada** se resuelven junto con el grupo; las marcadas como **Práctica independiente** las resuelven los participantes en sus propias copias de la notebook (`Archivo → Guardar una copia en Drive`).

## 🎯 Objetivos de aprendizaje

Al finalizar este tema, el participante será capaz de:

- Explicar la diferencia entre `plotly.express` (alto nivel) y `plotly.graph_objects` (bajo nivel).
- Construir gráficos interactivos (línea, barra, dispersión, mapa) con zoom, hover y leyenda interactiva.
- Añadir animación temporal (`animation_frame`) para explorar cómo cambian los datos a través del tiempo.
- Construir un mapa coroplético interactivo (`choropleth`) a nivel país.
- Exportar una figura interactiva como archivo `.html` autocontenido para compartir sin necesidad de Python.

## 🧠 Contenido teórico

### 1. De lo estático a lo interactivo

Hasta ahora, cada gráfico de Matplotlib/Seaborn es una **imagen**: una vez generada, no se puede hacer zoom, filtrar leyenda o pasar el cursor para ver el valor exacto. **Plotly** genera gráficos **interactivos basados en JavaScript (biblioteca D3.js/WebGL por debajo)**, que funcionan directamente en el navegador — perfectos para notebooks, dashboards y páginas web.

### 2. Dos formas de usar Plotly

| API | Cuándo usarla |
|---|---|
| **`plotly.express` (`px`)** | La mayoría de los casos: una función por tipo de gráfico (`px.line`, `px.bar`, `px.scatter`, `px.choropleth`...), recibe un DataFrame directamente. Análoga a Seaborn pero interactiva. |
| **`plotly.graph_objects` (`go`)** | Control fino: combinar múltiples trazas de tipos distintos, anotaciones complejas, dashboards personalizados. `px` internamente genera objetos `go.Figure`. |

En este taller usaremos principalmente `plotly.express`, y tocaremos `graph_objects` para personalización puntual.

### 3. Interactividad "gratis"

Cada figura de Plotly incluye, sin código adicional:
- **Zoom** (arrastrar para acercar, doble clic para restablecer).
- **Hover** con tooltips (puedes personalizar qué columnas mostrar con `hover_data`).
- **Leyenda interactiva**: clic para ocultar/mostrar series, doble clic para aislar una.
- **Exportación** a PNG desde la barra de herramientas de la figura.

### 4. Animaciones y mapas

- `animation_frame="anio"` convierte cualquier gráfico `px` en una animación con controles de reproducción — ideal para mostrar evolución temporal sin saturar un solo gráfico.
- `px.choropleth(..., locations="pais", locationmode="country names", color="variable")` genera mapas donde el color de cada país representa una variable — muy usado en visualización de datos globales (turismo, economía, salud pública, etc.).

### 5. Compartir sin Python

`fig.write_html("grafico.html")` genera un archivo HTML **autocontenido** (incluye la librería JS embebida) que cualquier persona puede abrir con un navegador, sin instalar nada — ideal para el repositorio de GitHub del taller.

## ⚙️ Configuración del entorno

Cargaremos `tema04_turismo_internacional.csv`: llegadas de turistas e ingresos por turismo (sintéticos) por país, mes y año (2022-2024).

In [ ]:
# === Carga del dataset ===
# Opción 1 (recomendada una vez publicado el repositorio del taller):
# reemplaza <usuario>/<repositorio> por la ruta real de tu repo de GitHub
# y ejecuta esta celda. Usa el botón "Raw" de GitHub para obtener la URL.
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/<usuario>/<repositorio>/main/"
    "datasets/tema04_turismo_internacional.csv"
)

import pandas as pd

try:
    df = pd.read_csv(GITHUB_RAW_URL)
    print("Datos cargados desde GitHub ✅  ->", df.shape)
except Exception as e:
    print("No se pudo leer desde GitHub todavía (repo no configurado o sin internet).")
    print("Sube manualmente el archivo 'tema04_turismo_internacional.csv' cuando se te solicite.")
    try:
        from google.colab import files
        subido = files.upload()  # selecciona tema04_turismo_internacional.csv
        df = pd.read_csv(list(subido.keys())[0])
    except ImportError:
        # Fuera de Colab (por ejemplo, ejecución local de prueba):
        df = pd.read_csv("tema04_turismo_internacional.csv")

df.head()

## 🧭 Práctica guiada

### Paso 1 · Línea interactiva — evolución temporal

In [ ]:
import plotly.express as px

df["fecha"] = pd.to_datetime(df["anio"].astype(str) + "-" + df["mes"].astype(str) + "-01")

paises_top = ["México", "España", "Francia", "Japón"]
datos_linea = df[df["pais"].isin(paises_top)].sort_values("fecha")

fig = px.line(
    datos_linea, x="fecha", y="llegadas_turistas_millones", color="pais",
    title="Llegadas de turistas internacionales (millones/mes)",
    labels={"llegadas_turistas_millones": "Turistas (millones)", "fecha": "Fecha", "pais": "País"},
)
fig.update_layout(hovermode="x unified")
fig.show()

**En vivo:** mostrar cómo hacer zoom sobre un rango de fechas, ocultar un país haciendo clic en la leyenda, y aislar uno con doble clic.

### Paso 2 · Barras — comparación entre países (con animación temporal)

In [ ]:
resumen_anual = (
    df.groupby(["pais", "continente", "anio"], as_index=False)["ingresos_turismo_usd_millones"]
    .sum()
)

fig = px.bar(
    resumen_anual.sort_values(["anio", "ingresos_turismo_usd_millones"]),
    x="ingresos_turismo_usd_millones", y="pais", color="continente",
    animation_frame="anio", orientation="h",
    title="Ingresos por turismo por país (animado por año)",
    labels={"ingresos_turismo_usd_millones": "Ingresos (millones USD)", "pais": "País"},
    range_x=[0, resumen_anual["ingresos_turismo_usd_millones"].max() * 1.1],
)
fig.show()

### Paso 3 · Dispersión con tamaño y animación

In [ ]:
fig = px.scatter(
    resumen_anual, x="ingresos_turismo_usd_millones", y="pais",
    size="ingresos_turismo_usd_millones", color="continente",
    animation_frame="anio", size_max=40,
    title="Ingresos por turismo (tamaño = magnitud), animado por año",
)
fig.show()

### Paso 4 · Mapa coroplético mundial

In [ ]:
mapa_datos = df[df["anio"] == 2024].groupby(["pais", "continente"], as_index=False)[
    ["llegadas_turistas_millones", "ingresos_turismo_usd_millones"]
].sum()

fig = px.choropleth(
    mapa_datos, locations="pais", locationmode="country names",
    color="llegadas_turistas_millones", hover_name="pais",
    hover_data={"ingresos_turismo_usd_millones": ":.0f"},
    color_continuous_scale="Plasma",
    title="Llegadas de turistas por país (2024, acumulado anual)",
)
fig.show()

### Paso 5 · Exportar a HTML autocontenido

In [ ]:
fig.write_html("mapa_turismo_2024.html", include_plotlyjs="cdn")
print("Archivo 'mapa_turismo_2024.html' generado. Se puede abrir directamente en cualquier navegador.")

## ✍️ Práctica independiente

**Ejercicio 1.** Crea un `px.line` mostrando `ingresos_turismo_usd_millones` a través del tiempo (`fecha`) para los países de **Sudamérica** presentes en el dataset (`Brasil`)  y algún país africano (`Egipto`, `Sudáfrica`), coloreado por país.

In [ ]:
# TODO: tu código aquí

**Ejercicio 2.** Usando `resumen_anual`, construye un `px.bar` (sin animación) que muestre solamente el año **2024**, ordenado de mayor a menor ingreso.

In [ ]:
# TODO: tu código aquí

**Ejercicio 3.** Construye un `px.sunburst` (o `px.treemap`) con jerarquía `continente → pais`, usando como valor `llegadas_turistas_millones` del año 2024.

In [ ]:
# TODO: tu código aquí

**Ejercicio 4 (reto).** Construye un `px.choropleth` animado (`animation_frame='anio'`) mostrando `ingresos_turismo_usd_millones` por país a través de los 3 años disponibles.

In [ ]:
# TODO: tu código aquí

---
### ✅ Soluciones (referencia para el profesor)

In [ ]:
# Ejercicio 1
paises_sel = ["Brasil", "Egipto", "Sudáfrica"]
sub = df[df["pais"].isin(paises_sel)].sort_values("fecha")
fig1 = px.line(sub, x="fecha", y="ingresos_turismo_usd_millones", color="pais",
               title="Ingresos por turismo - Brasil, Egipto y Sudáfrica")
fig1.show()

# Ejercicio 2
datos_2024 = resumen_anual[resumen_anual["anio"] == 2024].sort_values(
    "ingresos_turismo_usd_millones", ascending=True
)
fig2 = px.bar(datos_2024, x="ingresos_turismo_usd_millones", y="pais", color="continente",
              orientation="h", title="Ingresos por turismo 2024")
fig2.show()

# Ejercicio 3
datos_2024_full = df[df["anio"] == 2024].groupby(
    ["continente", "pais"], as_index=False
)["llegadas_turistas_millones"].sum()
fig3 = px.sunburst(datos_2024_full, path=["continente", "pais"],
                    values="llegadas_turistas_millones",
                    title="Llegadas de turistas 2024 por continente y país")
fig3.show()

# Ejercicio 4
fig4 = px.choropleth(
    resumen_anual, locations="pais", locationmode="country names",
    color="ingresos_turismo_usd_millones", animation_frame="anio",
    hover_name="pais", color_continuous_scale="Viridis",
    title="Ingresos por turismo por país (animado por año)",
)
fig4.show()

## 🔎 Cierre y puente al siguiente tema

Plotly nos da gráficos interactivos individuales de gran calidad. Pero, ¿qué pasa si queremos que el usuario **elija** qué país o variable ver, con controles como menús desplegables, sin tener que regenerar el notebook? Eso es exactamente lo que resuelve **Dash** (Tema 05): aplicaciones web interactivas construidas sobre Plotly.

## 📚 Recursos adicionales

- [Documentación oficial de Plotly (Python)](https://plotly.com/python/)
- [Plotly Express — referencia completa](https://plotly.com/python/plotly-express/)
- [Galería de gráficos de Plotly](https://plotly.com/python/basic-charts/)
- [Mapas coropléticos con Plotly](https://plotly.com/python/choropleth-maps/)
- [Animaciones con Plotly Express](https://plotly.com/python/animations/)